## **Feature Engineering**

En este notebook se realiza la selección, limpieza y encoding de las variables 
que serán utilizadas en el entrenamiento del modelo.

In [ ]:
import pandas as pd

df_train = pd.read_csv('data/train_transaction.csv')

df_train.shape

## Ordenar dataset por TransactionDT para luego separar el dataset en splits

In [ ]:
df_train = df_train.sort_values('TransactionDT').reset_index(drop=True)

In [ ]:
y = df_train['isFraud']
X = df_train.drop(columns=['isFraud'])

print(X.shape)
print(y.shape)

In [ ]:
X.dtypes.value_counts()

In [ ]:
X.select_dtypes(include='object').columns

M1 a M9 — son columnas de match, sus valores son solo "T" o "F" (verdadero/falso)



In [ ]:
X[['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']].value_counts()

In [ ]:
X['M4'].value_counts()

## Tipos de columnas categóricas

### Grupo M, columnas de match
- `M1, M2, M3, M5, M6, M7, M8, M9` → binarias (T/F) → mapeo simple: T=1, F=0
- `M4` → 3 valores (M0, M1, M2) → necesita encoding separado

### Grupo principal, categorías reales
- `ProductCD`, `card4`, `card6` → pocas categorías, valores de negocio
- `P_emaildomain`, `R_emaildomain` → dominios de email, muchos valores únicos posibles

In [ ]:
X[['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']] = X[['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']].apply(lambda x: x.map({'T': 1, 'F': 0}))

In [ ]:
X['M4'].value_counts()

## Encoding de M4

`M4` tiene 3 valores: `M0`, `M1`, `M2`. Se aplica **ordinal encoding** (M0→0, M1→1, M2→2) por dos razones:

1. El nombre de los valores sugiere un índice implícito, Verizon los nombró con orden numérico.
2. One-hot encoding generaría columnas adicionales innecesarias para un modelo de árboles como LightGBM, que maneja ordinales bien sin perder información.

> Limitación: no sabemos si el orden real es M0 < M1 < M2. Si el modelo falla, revisar este encoding es un buen punto de partida.

In [ ]:
X['M4'] = X['M4'].map(({'M0':0, 'M1':1, 'M2':2}))

In [ ]:
X['M4'].value_counts()

In [ ]:
X[['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']].nunique()

In [ ]:
for col in ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']:
    print(f"\n{col}:")
    print(X[col].value_counts())

In [ ]:
X.select_dtypes(include='object').columns

## Split temporal vs. aleatorio

En datos financieros con componente de tiempo, el split debe ser **cronológico**, no aleatorio.

- Split aleatorio, mezcla datos futuros en el entrenamiento, lo que hace que el modelo aprenda patrones que en producción no llegan a existir (**temporal leakage**).
- Split cronológico, entrena con el pasado, y valida con el futuro.

> Regla: siempre se predice el futuro con datos del pasado. El split debe reflejar eso.

## **Dividir dataset**

Se divide el dataset teniendo ordenado todo por tiempo de la variable TransactionDT, para tener el contexto real de como se decta, no de manera aleatoria

In [ ]:
X = X.sort_values('TransactionDT')

corte = int(len(X)*0.8)

X_train = X.iloc[:corte]
y_train = y.iloc[:corte]

X_Val = X.iloc[corte:]
y_Val = y.iloc[corte:]

In [ ]:
X_train.shape, X_Val.shape, y_train.shape, y_Val.shape

## Decisión: Target Encoding después del split

**Técnica elegida:** Target encoding para P_emaildomain y R_emaildomain  
**Calculado sobre:** X_train / y_train únicamente, luego aplicado a X_val

**Por qué después del split:**  
El target encoding reemplaza cada categoría con la tasa de fraude promedio de esa categoría.  
Si se calcula antes del split, las etiquetas de validación contaminan  el cálculo

**Supuesto que asume:**  
La distribución de dominios de correo en producción será similar a la de train.  
Dominios nuevos recibirán el promedio global como fallback.

## **Encoding a target encoding a p_emaildomain**

In [ ]:
tasa_fraude_dominio = y_train.groupby(X_train['P_emaildomain']).mean()

print(tasa_fraude_dominio)

**protonmail cuenta con el porcentaje de fraude más alto, con un 46%**

In [ ]:
X_train['P_emaildomain'] = X_train['P_emaildomain'].map(tasa_fraude_dominio)

In [ ]:
X_Val['P_emaildomain'] = X_Val['P_emaildomain'].map(tasa_fraude_dominio).fillna(y_train.mean())

## **Encoding a target encoding a R_emaildomain**

In [ ]:
tasa_fraude_R_emaildomain = y_train.groupby(X_train['R_emaildomain']).mean()
print(tasa_fraude_R_emaildomain)

In [ ]:
X_train['R_emaildomain'] = X_train['R_emaildomain'].map(tasa_fraude_R_emaildomain)

In [ ]:
X_Val['R_emaildomain'] = X_Val['R_emaildomain'].map(tasa_fraude_R_emaildomain).fillna(y_train.mean())

**One_Hot_Encoding** campo de productCD, card4 y card6

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_encoded = encoder.fit_transform(X_train[['ProductCD', 'card4', 'card6']])

X_Val_encoded = encoder.transform(X_Val[['ProductCD', 'card4', 'card6']])

X_train.drop(columns=['ProductCD', 'card4', 'card6'], inplace=True)
X_Val.drop(columns=['ProductCD', 'card4', 'card6'], inplace=True)



In [ ]:
X_train = pd.concat([X_train.reset_index(drop=True), pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out())], axis=1)
X_Val = pd.concat([X_Val.reset_index(drop=True), pd.DataFrame(X_Val_encoded, columns=encoder.get_feature_names_out())], axis=1)

In [ ]:
X_train.shape, X_Val.shape, y_train.shape, y_Val.shape

In [ ]:
X_train.select_dtypes(include='object').columns

**Guardar dataset en formato parquet**

In [ ]:
import fastparquet 

X_train.to_parquet('data/X_train.parquet', engine='fastparquet')
y_train.to_frame().to_parquet('data/y_train.parquet', engine='fastparquet')
X_Val.to_parquet('data/X_Val.parquet', engine='fastparquet')
y_Val.to_frame().to_parquet('data/y_Val.parquet', engine='fastparquet')

In [ ]:
X_train = pd.read_parquet('data/X_train.parquet')
print(X_train.shape)
print(X_train.columns.tolist())

## **Conclusiones**

En este notebook se preparó el dataset para el entrenamiento del modelo, dejándolo limpio y consistente. El primer paso fue ordenar las transacciones por la variable TransactionDT y realizar el split de forma temporal, no aleatoria, para simular un escenario real de cómo un banco recibe y procesa sus transacciones, entrenando siempre con datos del pasado y validando con datos del futuro.

Para las variables M1, M2, M3, M5, M6, M7, M8 y M9 se aplicó un encoding binario, ya que estas columnas solo contaban con dos valores posibles, verdadero o falso, por lo que un mapeo simple a 1 y 0 era suficiente y no perdía información. La variable M4 se trató de forma distinta, ya que a diferencia de las anteriores contaba con tres valores categóricos, M0, M1 y M2, por lo que se decidió aplicar un encoding ordinal, suponiendo que el orden de los nombres reflejaba un orden real entre las categorías.

Para las variables P_emaildomain y R_emaildomain se aplicó un target encoding, reemplazando cada dominio de correo por la tasa de fraude promedio asociada a ese dominio, calculada únicamente sobre el set de entrenamiento. Esto se eligió porque ambas columnas tenían una gran cantidad de valores únicos, lo que hacía inviable un one hot encoding sin generar demasiadas columnas extras.

Por último, para ProductCD, card4 y card6 se aplicó un one hot encoding, ya que estas columnas representan categorías de negocio reales con pocos valores únicos y sin un orden implícito entre ellas, lo que hace que esta técnica sea la más adecuada sin agregar una dimensionalidad innecesaria al dataset.